In [ ]:
!pip install transformers torch --quiet

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')
print("Libraries imported successfully!")
print(f"PyTorch version : {torch.__version__}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Running on      : {device.upper()}")

In [ ]:
MODEL_NAME = "microsoft/DialoGPT-medium"
print("Loading model, please wait...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()
print("Model loaded!")

In [ ]:
def generate_response(user_input, chat_history_ids=None):

    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors='pt'
    ).to(device)

    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            bot_input_ids,
            attention_mask=attention_mask,
            max_length=1000,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.75,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
        )

    response_ids = output_ids[:, bot_input_ids.shape[-1]:]
    response_text = tokenizer.decode(response_ids[0], skip_special_tokens=True)

    return response_text, output_ids

In [ ]:
def run_chatbot():
    print("=" * 55)
    print("        DialoGPT Chatbot - Hugging Face")
    print("=" * 55)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    print("(type 'exit' or 'quit' to stop)\n")
    chat_history_ids = None
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ['exit', 'quit']:
            print("Chatbot: Goodbye! Have a great day!")
            break
        if not user_input:
            print("Chatbot: Please say something!")
            continue
        response, chat_history_ids = generate_response(user_input, chat_history_ids)
        if not response.strip():
            response = "Could you rephrase that? I didn't quite get it."
        print(f"Chatbot: {response}\n")
        # trim history to avoid hitting the token limit
        if chat_history_ids is not None and chat_history_ids.shape[-1] > 800:
            chat_history_ids = chat_history_ids[:, -800:]
run_chatbot()

In [ ]:
def demo_conversation(inputs):
    print("=" * 55)
    print("        DialoGPT Chatbot - Demo")
    print("=" * 55)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?\n")
    chat_history_ids = None
    for user_input in inputs:
        print(f"You    : {user_input}")
        if user_input.lower() in ['exit', 'quit']:
            print("Chatbot: Goodbye!")
            break
        response, chat_history_ids = generate_response(user_input, chat_history_ids)
        if not response.strip():
            response = "Could you rephrase that? I didn't quite get it."
        print(f"Chatbot: {response}\n")
        if chat_history_ids is not None and chat_history_ids.shape[-1] > 800:
            chat_history_ids = chat_history_ids[:, -800:]
sample_inputs = [
    "Hello",
    "What is Artificial Intelligence?",
    "Who created Python?",
    "What is machine learning?",
    "Thank you",
    "exit"
]
demo_conversation(sample_inputs)